# 82 — ColBERT conversational retrieval: zero-shot pool-rerank probe (P1 kill-gate)

Mirrors nb74's recall harness. **Phase 1 (plan §9):** use off-the-shelf
`colbert-ir/colbertv2.0` as a *pool reranker* over the config-200 union pool
(`cs`) on the **dev turn-1 subset** — no PLAID index, no fine-tune. 

**Kill-gate:** ColBERT must beat single-vector dense on **turn-1 wall recall@20**.
If it does not, late interaction does not transfer to this domain → STOP, bank
config 200. Cells 1/3/4/5 are copied verbatim from nb74; the new work is the
`#82-probe` and `#82-gate` cells.

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in
# (datasets/transformers import JAX transitively; it grabs ~75% VRAM on first use).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive. retrieval_v2 holds sasrec/, lgbm/,
# ctx_cache/; dense holds the Qwen query/catalog cache for dense_metadata_qwen3.
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Deps: retrieval stack + ColBERT late-interaction (pylate). pylate is the only
# addition vs nb74 (lazy-imported by mcrs.retrieval_modules.colbert_late).
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0,<5' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn' \
    'omegaconf' 'pyyaml' 'pylate>=1.1.0'

In [ ]:
# 1b) Version guard — prove this runtime is on the LATEST pushed code BEFORE running
# anything. Catches (a) a stale clone and (b) stale in-kernel imports (a live kernel
# caches the OLD module even after a fresh clone). If this fails: Runtime -> Restart
# session, then re-run cell 1, then this cell.
import sys, subprocess
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sha = subprocess.check_output(['git','-C','/content/recsys2026','rev-parse','--short','HEAD']).decode().strip()
print('repo HEAD =', sha, '(Stage-A fixes = 572e1bb or later)')
from mcrs.retrieval_modules.colbert_late import DEFAULT_Q_LEN, strip_track_id_prefix
assert DEFAULT_Q_LEN == 96, f'STALE CODE (DEFAULT_Q_LEN={DEFAULT_Q_LEN}); Restart session + re-run cell 1'
assert strip_track_id_prefix('track_id: u, track_name: y') == 'track_name: y', 'STALE colbert_late'
print('OK: Stage-A fixes loaded (q_len=96 + strip_track_id_prefix). Safe to proceed.')

## Stage 1 — recall harness (nb74 parity): dev set + config-200 union pool `cs`

In [ ]:
# 3) Ensure the content-fused SASRec checkpoint exists (sasrec_v1). Trains it
# only if missing (it persists on the Drive cache across runtimes).
import os, sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
CACHE_DIR = '/content/recsys2026/experiments/cache'
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
SASREC_CKPT = f'{CACHE_DIR}/retrieval_v2/sasrec/sasrec_v1/sasrec.pt'
if os.path.exists(SASREC_CKPT):
    print('[sasrec] checkpoint present, skipping train:', SASREC_CKPT)
else:
    print('[sasrec] training content-fused SASRec (sasrec_v1, ~10 epochs)...')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out sasrec_v1 --epochs 10

In [ ]:
# 4) Build the FULL dev eval set + the config-200 union pool `cs` (verbatim nb74).
# A2: query appends listener_goal, matching the production crs_baseline query.
import numpy as np
import pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
goal_categories, goal_specificities, turn_numbers = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    goal_txt = (goal.get('listener_goal') or '').strip()  # A2: prod query includes this
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        _q = chr(10).join(lines)
        if goal_txt:
            _q = _q + chr(10) + 'goal: ' + goal_txt  # A2: train/serve parity
        queries.append(_q)
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        goal_categories.append(goal.get('category'))
        goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns')

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== config-200 union+SASRec (FULL dev, n=' + str(len(golds)) + ') ===')
print('  recall @20=' + str(round(recall_at(cs, 20), 4)) + ' @100=' + str(round(recall_at(cs, 100), 4)))

In [ ]:
# 5) Turn-1 / WALL recall@20 instrument (nb74 cell-6 helpers). The binding metric
# for Blind (100% turn-1, ~99% new-artist wall) is TURN-1 recall@20 over wall golds.
import numpy as np
from mcrs.eval_ndcg import recall_by_turn

def _artist_of(tid):
    md = item_db.metadata_dict.get(tid) or {}
    a = md.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

is_wall = np.array([_artist_of(g) not in {_artist_of(t) for t in p}
                    for g, p in zip(golds, played)])
turn1 = np.array([t == 1 for t in turn_numbers])
print(f'[strat] turns={len(golds)} turn1={int(turn1.sum())} wall={int(is_wall.sum())} '
      f'turn1&wall={int((turn1 & is_wall).sum())}')

def _recall_mask(cands, k, mask):
    idx = np.where(mask)[0]
    if len(idx) == 0: return float('nan')
    return float(np.mean([1.0 if golds[i] in cands[i][:k] else 0.0 for i in idx]))

def strat_recall(cands, label=''):
    rep = recall_by_turn(cands, golds, turn_numbers, k=20)
    t1 = rep.get('turn1'); w1 = _recall_mask(cands, 20, turn1 & is_wall)
    print(f'  {label:34s} recall@20 overall={rep["overall"]:.4f} '
          f'turn1={(t1 if t1 is not None else float("nan")):.4f} turn1&wall={w1:.4f}')
    return rep

## Stage 2 — ColBERT zero-shot pool-rerank probe (P1)

Off-the-shelf `colbert-ir/colbertv2.0` reranks each turn-1 query's config-200
pool (`cs[i]`) by MaxSim. Doc token embeddings are built once for every track id
that appears in a turn-1 pool (no full-catalog index needed). The query fed to
ColBERT is the *same* query single-vector dense gets — so this isolates **late
interaction vs averaging**.

In [ ]:
# 82-probe) ColBERT zero-shot pool-rerank over the config-200 pool, turn-1 subset.
# Requires cell 4 (queries, cs, item_db, turn_numbers, golds) + cell 5 (masks).
# Caveat (plan §11): colbertv2.0 is MS-MARCO/web-trained; a flat result conflates
# 'late interaction' with domain shift — Phase 2 fine-tune disambiguates. A clear
# win here is decisive; a flat result is a soft (not hard) kill.
import numpy as np
from mcrs.retrieval_modules.colbert_late import (ColbertRetriever, DEFAULT_COLBERT_MODEL,
                                                 strip_track_id_prefix)

# Restrict the probe to turn-1 rows (the Blind proxy; plan §9 P1).
t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
print(f'[probe] turn-1 rows: {len(t1_idx)}')

# Build ColBERT doc token-embeddings ONCE for every tid in a turn-1 pool.
pool_tids = sorted({tid for i in t1_idx for tid in cs[i]})
# Strip the leading 'track_id: <uuid>' (RCA #4: UUID hex dilutes MaxSim).
pool_texts = [strip_track_id_prefix(item_db.id_to_metadata(t)) for t in pool_tids]
print(f'[probe] unique pool docs to encode: {len(pool_tids)}')

retr = ColbertRetriever(doc_embs={}, model_name=DEFAULT_COLBERT_MODEL, q_len=96, d_len=96)
retr.encode_docs(pool_tids, pool_texts)  # one-time GPU encode (~minutes)

q_t1 = [queries[i] for i in t1_idx]
pools_t1 = [cs[i] for i in t1_idx]
colbert_t1 = retr.batch_rerank_pool(q_t1, pools_t1, topk=100)

# Splice reranked turn-1 lists back into a full-length cands array (non-turn-1 rows
# keep cs ordering) so the cell-5 strat_recall helpers apply unchanged.
colbert_cands = list(cs)
for j, i in enumerate(t1_idx):
    colbert_cands[i] = colbert_t1[j]
print('[probe] reranked', len(t1_idx), 'turn-1 pools')

# Reranker headroom: recall@20 can never exceed the pool's recall@100 (RCA #3).
print(f'[probe] turn-1&wall cs recall@100 (rerank CEILING) = {_recall_mask(cs, 100, turn1 & is_wall):.4f}  recall@20 = {_recall_mask(cs, 20, turn1 & is_wall):.4f}')

In [ ]:
# 82-gate) KILL-GATE: ColBERT vs single-vector dense vs union on turn-1 wall recall@20.
# Dense baseline = the dense_metadata_qwen3_instruct channel standalone (the single
# vector ColBERT must beat). Union (cs) = current config-200 ordering. ColBERT only
# reranks the cs pool, so its recall@100 == cs's by construction; the lift is purely
# @20 (in-pool golds in the rank 21-100 band lifted into top-20 — plan §0).
dense_only = load_retrieval_module('dense_metadata_qwen3_instruct', ITEM_DB, ['all_tracks'],
                                   CORPUS, CACHE_DIR, extra_config={})
dense_cands = dense_only.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids)
# (dense is content-only; it takes no batch_context, unlike the RRF union)

print('=== TURN-1 WALL RECALL@20 GATE (n_turn1=' + str(int(turn1.sum())) +
      ', n_turn1&wall=' + str(int((turn1 & is_wall).sum())) + ') ===')
for label, cands in [('single-vector dense', dense_cands),
                     ('union+SASRec (cs)', cs),
                     ('ColBERT rerank (cs pool)', colbert_cands)]:
    strat_recall(cands, label)

# Verdict on the binding metric.
dense_w = _recall_mask(dense_cands, 20, turn1 & is_wall)
cb_w = _recall_mask(colbert_cands, 20, turn1 & is_wall)
print()
print(f'[gate] turn1&wall recall@20: dense={dense_w:.4f}  colbert={cb_w:.4f}  '
      f'delta={cb_w - dense_w:+.4f}')
if cb_w > dense_w:
    print('[gate] PASS — late interaction beats single-vector dense. Proceed to P2',
          '(fine-tune music-colbert-v1 + PLAID index).')
else:
    print('[gate] FAIL/SOFT — see caveat (domain shift). If Phase-2 fine-tune is not',
          'at least neutral, STOP and bank config 200 (plan §9 kill-rule).')

## Stage 3 — P2: fine-tune `music-colbert-v1` + re-probe (tightened gate ladder)

P1 was a soft pass (ColBERT 0.258 > dense 0.215, but < union 0.325). P2 pays the
domain-shift tax: fine-tune on train-split MOVES_TOWARD_GOAL triples, then re-run
the **exact same pool-rerank probe**. **New bar = beat the union's 0.325** (not
dense's 0.215). Build the expensive PLAID full-catalog index only if this clears it.

> **RCA fixes applied (2026-06-11, wf8ii499r) — the earlier 0.281<0.325 was an INVALID kill-test.** Now: (1) turn-1 training positives are synthesized (were 100% filtered out by MOVES_TOWARD_GOAL); (2) q_len=96 keeps the `goal:` facet (q_len=32 truncated it off ~95% of queries); (3) doc text strips the `track_id:` UUID prefix. The reprobe is now a *fair* rerank test — but a rerank win/loss still isn't the recall thesis (that needs Stage B: the PLAID full-catalog index).

In [ ]:
# 82-build-data) P2 step 1: build ColBERT fine-tune triples (TRAIN split,
# MOVES_TOWARD_GOAL only, hard negs = SASRec-free pool non-golds). One-time;
# persists to the Drive cache. Use --max-rows 2000 first for a smoke build.
TRAIN_JSONL = f'{CACHE_DIR}/retrieval_v2/colbert_train.jsonl'
!cd /content/recsys2026 && python -u scripts/build_colbert_train_data.py \
    --output {TRAIN_JSONL} --cache-dir {CACHE_DIR} --pool-size 100 --k-negs 15 --max-rows 0
import os
print('[build-data] exists:', os.path.exists(TRAIN_JSONL),
      '| lines:', sum(1 for _ in open(TRAIN_JSONL)) if os.path.exists(TRAIN_JSONL) else 0)

In [ ]:
# 82-dev-eval-pack) Build the dev validation pack the trainer selects on. SAME
# turn-1 pool/golds as #82-reprobe, so in-loop model selection optimizes the EXACT
# gate metric (never a train-internal loss - the campaign's anti-overfit rule).
# Requires cells 4,5,7 (q_t1, pools_t1, t1_idx, golds, is_wall, item_db).
import pickle, os
from mcrs.retrieval_modules.colbert_late import strip_track_id_prefix
golds_t1 = [golds[i] for i in t1_idx]
wall_t1 = [bool(is_wall[i]) for i in t1_idx]
need = sorted({t for pool in pools_t1 for t in pool})
pack = {'queries': q_t1, 'pools': pools_t1, 'golds': golds_t1, 'wall': wall_t1,
        'tid_to_text': {t: strip_track_id_prefix(item_db.id_to_metadata(t)) for t in need}}
DEV_PACK = f'{CACHE_DIR}/retrieval_v2/colbert_dev_eval.pkl'
os.makedirs(os.path.dirname(DEV_PACK), exist_ok=True)
pickle.dump(pack, open(DEV_PACK, 'wb'))
print('[dev-pack]', len(pack['queries']), 'turn-1 queries,', len(need), 'pool docs ->', DEV_PACK)

In [ ]:
# 82-finetune) P2 step 2: warm-start colbertv2.0 -> music-colbert-v1 (PyLate
# contrastive). Validates on the dev pack every --eval-steps and saves the SINGLE
# best checkpoint by turn-1 recall@20 (overrides). --epochs 2 gives selection room.
COLBERT_OUT = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'
!cd /content/recsys2026 && python -u scripts/train_colbert.py \
    --train-jsonl {TRAIN_JSONL} --dev-eval-pack {DEV_PACK} --out-dir {COLBERT_OUT} \
    --epochs 2 --batch-size 32 --eval-steps 500 --dev-subset 300
print('[finetune] best model saved to', COLBERT_OUT)

In [ ]:
# 82-reprobe) P2 GATE: re-run the EXACT pool-rerank probe with FINE-TUNED
# music-colbert-v1. New bar = beat the UNION's turn-1 wall recall@20, not just
# dense. Reuses cell-5 masks + #82-probe vars (pool_tids, pool_texts, q_t1,
# pools_t1, t1_idx, colbert_cands). Build the PLAID index ONLY if this clears the union.
retr_ft = ColbertRetriever(doc_embs={}, model_name=COLBERT_OUT, q_len=96, d_len=96)
retr_ft.encode_docs(pool_tids, pool_texts)   # same turn-1 pool docs as zero-shot
colbert_ft_t1 = retr_ft.batch_rerank_pool(q_t1, pools_t1, topk=100)
colbert_ft_cands = list(cs)
for j, i in enumerate(t1_idx):
    colbert_ft_cands[i] = colbert_ft_t1[j]

print('=== P2 RE-PROBE: fine-tuned vs union vs zero-shot (turn-1 wall recall@20) ===')
for label, cands in [('union+SASRec (cs)', cs),
                     ('ColBERT zero-shot', colbert_cands),
                     ('ColBERT fine-tuned', colbert_ft_cands)]:
    strat_recall(cands, label)

union_w = _recall_mask(cs, 20, turn1 & is_wall)
ft_w = _recall_mask(colbert_ft_cands, 20, turn1 & is_wall)
print()
print(f'[P2 gate] turn1&wall recall@20: union={union_w:.4f}  colbert_ft={ft_w:.4f}  '
      f'delta={ft_w - union_w:+.4f}')
if ft_w > union_w:
    print('[P2 gate] PASS - fine-tuned ColBERT beats the union it reranks. Build the',
          'PLAID full-catalog index (P2 step 3): it can surface NEW golds, not just reorder.')
else:
    print('[P2 gate] FAIL (rerank) - fine-tuned ColBERT below the union on the FAIR test.',
          'Reranking is not the win; the recall thesis still needs Stage B (build the PLAID',
          'full-catalog index, gate on union recall@100 vs 0.495). Only a flat full-catalog',
          'recall@100 justifies banking config 200.')

## Stage 4 — Stage B: full-catalog ColBERT recall channel (the REAL recall thesis)

The reranker (Stage A) tied the union — but a reranker can only reorder the union's
own top-100; it can NEVER add a gold the union missed (recall@100 is invariant). This
is the only test that answers the plan's actual question: **run music-colbert-v1 as a
full-catalog retriever (PLAID over 47k) — does it surface new-artist/wall golds the
union's BM25+dense+session channels never reached, raising recall@100 above cs 0.495?**
A flat result here is the genuine, decisive refutation (then bank config 200); a rise
is real new recall — the thing the whole campaign needs.

In [ ]:
# 82-build-index) STAGE B step 1: build the PyLate PLAID index over the full 47k
# catalog with the fine-tuned music-colbert-v1 (UUID-stripped docs). One-time, GPU.
COLBERT_OUT = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'  # the trained model (on Drive)
INDEX_FOLDER = f'{CACHE_DIR}/retrieval_v2/colbert/plaid'
!cd /content/recsys2026 && python -u scripts/build_colbert_index.py \
    --model-dir {COLBERT_OUT} --index-folder {INDEX_FOLDER} --index-name colbert-music-v1
print('[build-index] PLAID index ->', INDEX_FOLDER)

In [ ]:
# 82-recall-gate) STAGE B — the REAL thesis: does ColBERT as a FULL-CATALOG retriever
# surface turn-1 wall golds the union MISSED? A reranker can't (recall@100 invariant);
# this can. Requires #82-build-index + cells '# 4)' (queries, turn_numbers, cs, golds)
# and '# 5)' (is_wall). t1_idx/q_t1/COLBERT_OUT are self-defined below -> no manual snippet.
import numpy as np
from mcrs.retrieval_modules.colbert_late import ColbertIndexRetriever
COLBERT_OUT = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'  # trained model (on Drive)
t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
q_t1 = [queries[i] for i in t1_idx]

idx_retr = ColbertIndexRetriever(index_folder=f'{CACHE_DIR}/retrieval_v2/colbert/plaid',
                                 index_name='colbert-music-v1', model_name=COLBERT_OUT)
colbert_full = idx_retr.batch_text_to_item_retrieval(q_t1, topk=100)  # full 47k catalog

golds_t1 = [golds[i] for i in t1_idx]
wall_t1 = np.array([bool(is_wall[i]) for i in t1_idx])
cs_t1 = [cs[i] for i in t1_idx]

def _rec(cands, k, mask):
    rows = np.where(mask)[0]
    return float(np.mean([1.0 if golds_t1[i] in cands[i][:k] else 0.0 for i in rows])) if len(rows) else float('nan')

cs100 = _rec(cs_t1, 100, wall_t1)
cb100 = _rec(colbert_full, 100, wall_t1)
# Combined reachability = gold in cs@100 OR colbert@100 — the UPPER BOUND on what any
# fusion of the two could reach. If combined ~ cs100, fusion cannot help (clean FAIL).
combined = [list(dict.fromkeys(list(cs_t1[j][:100]) + list(colbert_full[j][:100]))) for j in range(len(q_t1))]
comb = _rec(combined, 10**9, wall_t1)
# Rescue: of wall golds the union MISSED, how many does ColBERT full-catalog find?
wi = np.where(wall_t1)[0]
missed = [j for j in wi if golds_t1[j] not in cs_t1[j][:100]]
rescued = [j for j in missed if golds_t1[j] in colbert_full[j][:100]]
print(f'[Stage B] turn-1&wall recall@100: union cs={cs100:.4f}  ColBERT full-catalog={cb100:.4f}')
print(f'[Stage B] combined (cs OR colbert) reachability@100 = {comb:.4f}  (ceiling rise vs cs = {comb-cs100:+.4f})')
print(f'[Stage B] wall golds MISSED by union: {len(missed)}; RESCUED by ColBERT: {len(rescued)} '
      f'({100*len(rescued)/max(len(missed),1):.1f}%)')
if comb - cs100 > 0.01:
    print('[Stage B] PASS - ColBERT surfaces NEW golds the union missed. The recall ceiling',
          'rises -> build the fused colbert union channel (use_colbert in _wrrf_union_v1_specs).')
else:
    print('[Stage B] FAIL - ColBERT adds ~no new golds; the wall is unreachable by content',
          'retrieval. ColBERT genuinely refuted for this domain -> bank config 200.')

In [ ]:
# 82-fuse-gate) W3.a: fuse ColBERT into the union + SWEEP w_colbert (ONE retrieval, in-memory
# RRF re-fusion via fuse_per_sub). Turn-1 subset only = the gate population (~3 min, not 15).
# Gate on recall@20 (nDCG-relevant), NOT recall@100. Requires #82-build-index + cells '# 4)'/'# 5)'.
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL
import numpy as np
INDEX_FOLDER = f'{CACHE_DIR}/retrieval_v2/colbert/plaid'
COLBERT_OUT = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'

t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
q_t1   = [queries[i] for i in t1_idx]
u_t1   = [user_ids[i] for i in t1_idx]
ctx_t1 = [ctx[i] for i in t1_idx]
golds_t1 = [golds[i] for i in t1_idx]
cs_t1  = [cs[i] for i in t1_idx]

fused = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0,
                  'use_colbert': True, 'w_colbert': 1.0,
                  'colbert_index_folder': INDEX_FOLDER, 'colbert_index_name': 'colbert-music-v1',
                  'colbert_model': COLBERT_OUT})
per_sub, labels = fused.batch_per_sub_rankings(q_t1, user_ids=u_t1, batch_context=ctx_t1)  # retrieve ONCE
print('[fuse] channels:', labels)

def recall(cands, k):
    return float(np.mean([1.0 if golds_t1[i] in cands[i][:k] else 0.0 for i in range(len(q_t1))]))
def w_for(lab, wc):  # union defaults: dense 0.7, everything else 1.0; colbert = swept
    return wc if 'colbert' in lab else (0.7 if 'dense' in lab else 1.0)

print(f'[fuse] cs baseline (turn-1&wall): recall@20={recall(cs_t1,20):.4f}  recall@100={recall(cs_t1,100):.4f}')
best = (None, -1.0)
for wc in [0.0, 0.3, 0.5, 0.7, 1.0]:  # 0.0 = union without colbert (sanity == cs)
    weights = [w_for(lab, wc) for lab in labels]
    fc = RRF_MODEL.fuse_per_sub(per_sub, weights, k=60, topk=100)
    r20, r100 = recall(fc, 20), recall(fc, 100)
    flag = '  <- best@20' if r20 > best[1] else ''
    if r20 > best[1]: best = (wc, r20)
    print(f'  w_colbert={wc:.1f}: recall@20={r20:.4f} (delta {r20-recall(cs_t1,20):+.4f})  recall@100={r100:.4f}{flag}')
print(f'[fuse] BEST w_colbert={best[0]} -> recall@20={best[1]:.4f}. Next (P3): run union+colbert',
      'through the reranker for turn-1 nDCG@20 vs 0.2990 (does the +recall@20 convert?).')

## Stage 5 — P3: does the fused recall@20 gain CONVERT to nDCG@20?

The decisive gate. Stage B + the fuse gave +0.022 turn-1 wall **recall@20** — but this
campaign's recurring trap is recall rising while **nDCG stays flat** (the reranker never
lifts the new golds into the top ranks). P3 A/Bs the SAME pipeline with vs without ColBERT,
turn-1: (1) recall-only nDCG@20 (fused RRF top-20, no model) and (2) LGBM-reranked nDCG@20.
PASS = union+colbert nDCG > union nDCG -> finalize config 202 + Blind.

In [ ]:
# 82-p3-ndcg) P3: nDCG@20 conversion A/B (union vs union+colbert), turn-1. ONE retrieval,
# two RRF fusions (w_colbert 0 vs best). recall-only nDCG always works; LGBM rerank is
# try/except (set LGBM_DIR to a model on your Drive). Requires #82-build-index + '# 4)'/'# 5)'.
import math, os
import numpy as np
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL

W_COLBERT = 1.0   # set to the best w_colbert from the #82-fuse-gate sweep
LGBM_DIR  = f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_v1'  # change to a model present on your Drive

t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
q_t1   = [queries[i] for i in t1_idx];  u_t1 = [user_ids[i] for i in t1_idx]
ctx_t1 = [ctx[i] for i in t1_idx];      golds_t1 = [golds[i] for i in t1_idx]
played_t1 = [played[i] for i in t1_idx]
gc_t1 = [goal_categories[i] for i in t1_idx];  gs_t1 = [goal_specificities[i] for i in t1_idx]

fused = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0, 'use_colbert': True, 'w_colbert': W_COLBERT,
                  'colbert_index_folder': f'{CACHE_DIR}/retrieval_v2/colbert/plaid',
                  'colbert_index_name': 'colbert-music-v1',
                  'colbert_model': f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'})
per_sub, labels = fused.batch_per_sub_rankings(q_t1, user_ids=u_t1, batch_context=ctx_t1)
def w_for(lab, wc): return wc if 'colbert' in lab else (0.7 if 'dense' in lab else 1.0)
pool_base = RRF_MODEL.fuse_per_sub(per_sub, [w_for(l, 0.0) for l in labels], 60, 100)        # union, NO colbert
pool_cb   = RRF_MODEL.fuse_per_sub(per_sub, [w_for(l, W_COLBERT) for l in labels], 60, 100)  # union + colbert

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds_t1):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2)
                break
    return s / len(golds_t1)

nb0, ncb0 = ndcg20(pool_base), ndcg20(pool_cb)
print(f'=== P3 turn-1 nDCG@20 (n={len(q_t1)}) - RECALL-ONLY (fused top-20, no reranker) ===')
print(f'  union          : {nb0:.4f}')
print(f'  union+colbert  : {ncb0:.4f}   (delta {ncb0-nb0:+.4f})')

try:
    from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
    sidx = labels.index('sasrec_seq')
    def rerank_ndcg(pool):
        efpc = []
        for qi, cands in enumerate(pool):
            rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
            efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
        esi = [{'played_tids': played_t1[i], 'turn_number': 1, 'prior_track_count': len(played_t1[i])}
               for i in range(len(q_t1))]
        rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=LGBM_DIR)
        return ndcg20(rr.rerank(q_t1, pool, topk=20, user_ids=u_t1, goal_categories=gc_t1,
                                goal_specificities=gs_t1, user_profiles_raw=[None]*len(q_t1),
                                extra_features_per_candidate=efpc, extra_session_info=esi))
    nbr, ncbr = rerank_ndcg(pool_base), rerank_ndcg(pool_cb)
    print(f'=== P3 turn-1 nDCG@20 - LGBM reranked ({os.path.basename(LGBM_DIR)}) ===')
    print(f'  union          : {nbr:.4f}')
    print(f'  union+colbert  : {ncbr:.4f}   (delta {ncbr-nbr:+.4f})')
    if ncbr > nbr:
        print('[P3] PASS - ColBERT CONVERTS to nDCG@20 through the reranker -> finalize config 202 + Blind.')
    else:
        print('[P3] recall@20 gain did NOT convert through the reranker (the campaign trap). Try w_colbert sweep / listwise reranker.')
except Exception as e:
    print(f'[P3] LGBM rerank skipped ({type(e).__name__}: {e}).')
    print('     Point LGBM_DIR at a model on your Drive (ls experiments/cache/retrieval_v2/lgbm/).')
    print('     The recall-only nDCG A/B above is the core signal regardless.')


## Stage 6 — P5: config 202 Blind-A submission (the real verdict)

config 202 = config 200 (union+SASRec+propose-ground -> LLM listwise -> v5-kto) + the
`use_colbert` channel at w_colbert=1.0. Runs the full stack on Blind-A (80 sessions) ->
prediction.json -> zip for CodaBench. dev->Blind is LOSSY (config 198's +0.05 dev went
flat), so this is the decisive test of whether the +0.0198 dev nDCG survives to Blind.

In [ ]:
# 82-blindA) P5: Blind-A inference for config 202 = union+SASRec+pg+ColBERT -> LLM listwise
# -> v5-kto responder. Requires: PLAID index on Drive (#82-build-index) + GEMINI_API_KEY.
# ~$0.1-0.3 Gemini (pg flash proposals + flash-lite listwise rerank), ~15-30 min for 80 sessions.
import os, json, zipfile, datetime
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), \
    'Set GEMINI_API_KEY in Colab Secrets (key icon, left sidebar) then re-run'

TID  = '202-union-sasrec-pg-colbert-llm-listwise-v5kto-blindA'
PRED = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index first'

import subprocess
subprocess.run(
    ['python', 'run_inference_blindset.py', '--tid', TID, '--batch_size', '8'],
    cwd='/content/recsys2026/music-crs-baselines', check=True)

preds = json.load(open(PRED))
assert len(preds) == 80, f'expected 80 Blind-A entries, got {len(preds)} -- DO NOT submit'
print('[blindA-202] entries:', len(preds), '| keys:', list(preds[0].keys()))

zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-202-colbert.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED, arcname='prediction.json')   # MUST be 'prediction.json' at zip ROOT for CodaBench
print('[blindA-202] submission zip ready:', zip_path)
print('[blindA-202] upload to CodaBench; then log the score via scripts/blind_a_score_tracker.py')


## Stage 7 — P4: CLAP text->audio recall channel (orthogonal to ColBERT)

The plan pairs ColBERT (text) with CLAP (audio) — CLAP reaches new-artist wall golds by
*sound*, a signal every text channel structurally lacks. This is the cold-firable
`clap_text` channel (query text -> CLAP audio space), distinct from `clap_recall` (which
mean-pools played tracks and is dead at turn-1). Gate: does +colbert+clap beat +colbert
on turn-1 wall recall@20? If yes -> the full-stack config 203 (ColBERT + CLAP).

In [ ]:
# 82-p4-clap) P4: does the COLD CLAP text->audio channel add recall ON TOP of ColBERT?
# CLAP reaches new-artist golds by ACOUSTICS (orthogonal to all text channels). ONE retrieval
# (union+sasrec+colbert+clap_text on turn-1), fuse 4 ways. Gate: does +colbert+clap beat
# +colbert on turn-1 wall recall@20? Requires #82-build-index + cells '# 4)'/'# 5)'.
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL
import numpy as np
INDEX_FOLDER = f'{CACHE_DIR}/retrieval_v2/colbert/plaid'
COLBERT_OUT  = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'

t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
q_t1   = [queries[i] for i in t1_idx];  u_t1 = [user_ids[i] for i in t1_idx]
ctx_t1 = [ctx[i] for i in t1_idx];      golds_t1 = [golds[i] for i in t1_idx]

u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0,
                  'use_colbert': True, 'w_colbert': 1.0,
                  'colbert_index_folder': INDEX_FOLDER, 'colbert_index_name': 'colbert-music-v1',
                  'colbert_model': COLBERT_OUT,
                  'use_clap_text': True, 'w_clap_text': 1.0})
per_sub, labels = u.batch_per_sub_rankings(q_t1, user_ids=u_t1, batch_context=ctx_t1)
print('[p4] channels:', labels)

def rec(cands, k):
    return float(np.mean([1.0 if golds_t1[i] in cands[i][:k] else 0.0 for i in range(len(q_t1))]))
def W(lab, wcol, wclap):
    if 'colbert' in lab: return wcol
    if 'clap_text' in lab: return wclap
    return 0.7 if 'dense' in lab else 1.0

print('arm              | recall@20 | recall@100')
for name, wcol, wclap in [('union (base)', 0.0, 0.0), ('+colbert', 1.0, 0.0),
                          ('+clap only', 0.0, 1.0), ('+colbert+clap', 1.0, 1.0)]:
    f = RRF_MODEL.fuse_per_sub(per_sub, [W(l, wcol, wclap) for l in labels], 60, 100)
    print(f'  {name:14s} |  {rec(f,20):.4f}  |   {rec(f,100):.4f}')
print('P4 gate: if +colbert+clap > +colbert on recall@20, CLAP adds orthogonal recall -> use config 203.')


In [ ]:
# 82-blindA-203) P5: Blind-A inference for config 203 = union+SASRec+pg+ColBERT+CLAP(text)
# -> LLM listwise -> v5-kto (the plan's FULL stack). Streams inference output so errors are
# visible. Requires PLAID index on Drive (#82-build-index) + GEMINI_API_KEY. ~15-30 min.
import os, json, zipfile, datetime
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), \
    'Set GEMINI_API_KEY in Colab Secrets (key icon) then re-run'
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index'

TID  = '203-union-sasrec-pg-colbert-claptext-llm-listwise-v5kto-blindA'
PRED = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -80
%cd /content/recsys2026

assert os.path.exists(PRED), 'inference did not write prediction.json -> read the traceback above'
preds = json.load(open(PRED))
assert len(preds) == 80, f'expected 80 Blind-A entries, got {len(preds)} -- DO NOT submit'
print('[blindA-203] entries:', len(preds), '| keys:', list(preds[0].keys()))
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-203-colbert-clap.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED, arcname='prediction.json')   # MUST be 'prediction.json' at zip ROOT for CodaBench
print('[blindA-203] submission zip ready:', zip_path)


In [ ]:
# 82-blindA-203-gemini) RESPONDER SWAP (nb80/81 known-good): apply the GEMINI responder
# (gemini-2.5-pro + best-of-3, flash judge, EXACT gemini_responder.py RESPONDER_INSTRUCTIONS
# prompt, plain/no structured-personality) on top of config 203's predicted_track_ids.
# track_ids UNTOUCHED; only predicted_response is regenerated. Requires #82-blindA-203 to have
# produced the 203 prediction.json + GEMINI_API_KEY. SMOKE=True does a 5-row check first.
import os, json, zipfile, datetime
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY in Colab Secrets'
os.environ['GEMINI_RESPONDER_MODEL'] = 'gemini-2.5-pro'   # nb80 known-good generator (~4.05)

TID      = '203-union-sasrec-pg-colbert-claptext-llm-listwise-v5kto-blindA'
PRED_IN  = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
PRED_OUT = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'
assert os.path.exists(PRED_IN), 'run #82-blindA-203 first to produce the 203 prediction.json'

SMOKE = False  # FULL 80 rows + package the submission zip (was True for a 5-row smoke check)
limit = '--limit 5' if SMOKE else ''
%cd /content/recsys2026
!python -u scripts/gemini_responder.py --cache-dir /content/drive/MyDrive/recsys2026_cache/responder --pred {PRED_IN} --out {PRED_OUT} \
    --dataset {BLIND_DATASET} --top-n 3 --best-of 3 --judge-model gemini-2.5-flash --sleep 0.2 {limit} 2>&1 | tail -40

out = json.load(open(PRED_OUT))
print('\nrows:', len(out))
print('sample response:\n', out[0]['predicted_response'][:400])
if SMOKE:
    print('\n*** SMOKE (5 rows). Set SMOKE=False + re-run, then it packages the zip. ***')
else:
    assert len(out) == 80, f'expected 80 rows, got {len(out)} -- DO NOT submit'
    zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-203-colbert-clap-gemini.zip'
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_OUT, arcname='prediction.json')   # 'prediction.json' at zip ROOT for CodaBench
    print('[blindA-203-gemini] submission zip ready:', zip_path)


In [ ]:
# 82-build-bge) ONE-TIME: embed the 47k catalog with bge-base-en-v1.5 for the use_bge channel
# (config 204). DENSE_LOCAL requires this precomputed pickle; ~few min GPU, persists on Drive.
# Run this BEFORE #82-blindA-204. Skip if you set use_bge:false in config 204.
import os
BGE_PKL = f'{CACHE_DIR}/dense_local/BAAI_bge-base-en-v1.5/bge-base-en-metadata/track_embeddings.pkl'
if os.path.exists(BGE_PKL):
    print('[build-bge] embeddings already present:', BGE_PKL)
else:
    !cd /content/recsys2026 && python scripts/embed_catalog.py --model BAAI/bge-base-en-v1.5 --label bge-base-en-metadata --cache-root {CACHE_DIR}/dense_local 2>&1 | tail -20
    print('[build-bge] done ->', BGE_PKL, '| exists:', os.path.exists(BGE_PKL))

In [ ]:
# 82-blindA-204) FINAL Blind-A eval: config 204 retrieval -> Gemini responder swap -> zip.
# Inference is SKIPPED if 204.json exists (so changing BEST_OF only re-runs the responder).
# Responder streams LIVE progress (no tail). Requires PLAID index + GEMINI_API_KEY.
import os, json, zipfile, datetime, subprocess
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY in Colab Secrets'
os.environ['GEMINI_RESPONDER_MODEL'] = 'gemini-2.5-pro'   # nb80 known-good responder
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index'

BEST_OF = 1   # 1 = single-shot pro (~4.05 LLM, ~15 min); 3 = best (~4.35, ~50 min). Change + re-run.

MCRS = '/content/recsys2026/music-crs-baselines'
TID  = '204-union-sasrec-pg-colbert-claptext-bge-intentstate-llm-listwise-v5kto-blindA'
PRED = f'{MCRS}/exp/inference/blindset_A/{TID}.json'
GEM  = f'{MCRS}/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# 1) full-stack retrieval -> predicted_track_ids (subprocess streams live; SKIP if already done)
if os.path.exists(PRED):
    print('[blindA-204] reusing existing track_ids:', PRED)
else:
    subprocess.run(['python', 'run_inference_blindset.py', '--tid', TID, '--batch_size', '8'],
                   cwd=MCRS, check=False)
assert os.path.exists(PRED), 'inference did not write 204.json -> read the traceback above'

# 2) Gemini responder swap (overwrites predicted_response; track_ids untouched) — LIVE progress
!cd /content/recsys2026 && python -u scripts/gemini_responder.py --cache-dir /content/drive/MyDrive/recsys2026_cache/responder --pred {PRED} --out {GEM} --dataset {BLIND_DATASET} --top-n 3 --best-of {BEST_OF} --judge-model gemini-2.5-flash --sleep 0.2

assert os.path.exists(GEM), 'gemini responder did not write output -> read the traceback above'
preds = json.load(open(GEM))
assert len(preds) == 80, f'expected 80 entries, got {len(preds)} -- DO NOT submit'
print('[blindA-204] entries:', len(preds), '| best_of:', BEST_OF)
print('sample response:\n', preds[0]['predicted_response'][:400])
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-204-full-gemini-bo{BEST_OF}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(GEM, arcname='prediction.json')   # 'prediction.json' at zip ROOT for CodaBench
print('[blindA-204] submission zip ready:', zip_path)


In [ ]:
# 82-blindA-205) EXP-006 CONFIRM: config 205 = config 203 + LLM-listwise ranker UPGRADED to
# gemini-2.5-flash @ max_output_tokens=2048 (dev: +0.0344 turn-1 nDCG vs flash-lite). top-n 1 +
# best-of 1 EXACTLY match the 0.4673 baseline (EXP-001), so the ONLY change vs 0.4673 is the ranker.
# Inference SKIPPED if 205.json exists. Requires PLAID index + GEMINI_API_KEY. ~15-30 min.
import os, json, zipfile, datetime, subprocess
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY in Colab Secrets'
os.environ['GEMINI_RESPONDER_MODEL'] = 'gemini-2.5-pro'   # known-good responder (matches the 0.4673 baseline)
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index'

BEST_OF = 1   # match the 0.4673 baseline (EXP-001 used bo1) so the ONLY diff vs it is the flash ranker

# FROZEN responder (2026-06-14): reuse the known-good 0.50 responses; only rows whose top-1
# track CHANGED hit Gemini. Set REUSE_FROM='' to regenerate all (when a better responder is ready).
REUSE_FROM = '/content/drive/MyDrive/recsys2026_submissions/2026-06-13-blindA-205-flashrank-gemini-bo1.zip'
reuse_flag = f'--reuse-from {REUSE_FROM}' if REUSE_FROM else ''

MCRS = '/content/recsys2026/music-crs-baselines'
TID  = '205-union-sasrec-pg-colbert-claptext-flashrank-llm-listwise-v5kto-blindA'
PRED = f'{MCRS}/exp/inference/blindset_A/{TID}.json'
GEM  = f'{MCRS}/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# 1) full-stack retrieval WITH the flash listwise ranker @2048 (config 205 wires reranker_max_output_tokens)
if os.path.exists(PRED):
    print('[blindA-205] reusing existing track_ids:', PRED)
else:
    subprocess.run(['python', 'run_inference_blindset.py', '--tid', TID, '--batch_size', '8'],
                   cwd=MCRS, check=False)
assert os.path.exists(PRED), 'inference did not write 205.json -> read the traceback above'

# 2) Gemini responder swap (track_ids untouched; top-n 1 + bo1 == the 0.4673 baseline) -- LIVE progress
!cd /content/recsys2026 && python -u scripts/gemini_responder.py --cache-dir /content/drive/MyDrive/recsys2026_cache/responder --pred {PRED} --out {GEM} --dataset {BLIND_DATASET} --top-n 1 --best-of {BEST_OF} {reuse_flag} --judge-model gemini-2.5-flash --sleep 0.2

assert os.path.exists(GEM), 'gemini responder did not write output -> read the traceback above'
preds = json.load(open(GEM))
assert len(preds) == 80, f'expected 80 entries, got {len(preds)} -- DO NOT submit'
print('[blindA-205] entries:', len(preds), '| best_of:', BEST_OF, '| ranker: gemini-2.5-flash @2048 tok')
print('sample response:\n', preds[0]['predicted_response'][:400])
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-205-flashrank-gemini-bo{BEST_OF}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(GEM, arcname='prediction.json')   # 'prediction.json' at zip ROOT for CodaBench
print('[blindA-205] submission zip ready:', zip_path)


In [ ]:
# 82-blindA-207) EXP-009/010 Blind: config 207 = e5-instruct dense REPLACEMENT (w=1.5) + reranker
# WINDOW k=100 (the EXP-010 win: +0.0128 dev turn-1 nDCG on the stripped union, driven by k=50->100
# letting the ranker see wall golds at union rank 51-100). Responder = gemini-2.5-flash-LITE (bo1,
# top-n 1). Straight-to-Blind per user call (SKIPS the full-union dev confirm; the +0.0128 is
# UNCONFIRMED on the union WITH ColBERT/CLAP/pg -> read the score against the ±0.05 Blind noise band:
# expected composite ~0.50 vs the 0.49 best, i.e. one submission can't statistically confirm it).
# Requires PLAID index + GEMINI_API_KEY. Encodes e5 catalog embeddings if missing (~5 min, fp16 G4).
import os, json, zipfile, datetime, subprocess
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY in Colab Secrets'
os.environ['GEMINI_RESPONDER_MODEL'] = 'gemini-2.5-flash-lite'   # lite responder (user decision, unvalidated)
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index'

BEST_OF = 1

# FROZEN responder (2026-06-14): reuse the known-good 0.50 responses; only rows whose top-1
# track CHANGED hit Gemini. Set REUSE_FROM='' to regenerate all (when a better responder is ready).
REUSE_FROM = '/content/drive/MyDrive/recsys2026_submissions/2026-06-13-blindA-205-flashrank-gemini-bo1.zip'
reuse_flag = f'--reuse-from {REUSE_FROM}' if REUSE_FROM else ''
MCRS = '/content/recsys2026/music-crs-baselines'
TID  = '207-union-sasrec-pg-colbert-claptext-e5replace-flashrank-llm-listwise-v5kto-blindA'
PRED = f'{MCRS}/exp/inference/blindset_A/{TID}.json'
GEM  = f'{MCRS}/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# 0) ensure e5 catalog embeddings exist (config 207's dense channel = multilingual-e5-large-instruct)
E5 = f'{CACHE_DIR}/dense_local/intfloat_multilingual-e5-large-instruct/e5-mli-metadata/track_embeddings.pkl'
if not os.path.exists(E5):
    print('[blindA-207] encoding e5 catalog embeddings (one-time, fp16, ~5 min)...')
    subprocess.run(['python', 'scripts/embed_catalog.py',
                    '--model', 'intfloat/multilingual-e5-large-instruct', '--label', 'e5-mli-metadata',
                    '--fields', 'track_name', 'artist_name', 'album_name', 'tag_list', 'release_date',
                    '--batch-size', '256', '--dtype', 'auto', '--max-seq-len', '256',
                    '--cache-root', f'{CACHE_DIR}/dense_local'], cwd='/content/recsys2026', check=True)
assert os.path.exists(E5), f'e5 embeddings not written to {E5}'

# 1) full-stack retrieval: union (e5 replace w1.5 + colbert + clap + pg) -> flash listwise ranker @ k=100
if os.path.exists(PRED):
    print('[blindA-207] reusing existing track_ids:', PRED)
else:
    subprocess.run(['python', 'run_inference_blindset.py', '--tid', TID, '--batch_size', '8'],
                   cwd=MCRS, check=False)
assert os.path.exists(PRED), 'inference did not write 207.json -> read the traceback above'

# 2) Gemini responder swap (flash-lite; track_ids untouched; top-n 1 + bo1) -- LIVE progress
!cd /content/recsys2026 && python -u scripts/gemini_responder.py --cache-dir /content/drive/MyDrive/recsys2026_cache/responder --pred {PRED} --out {GEM} --dataset {BLIND_DATASET} --top-n 1 --best-of {BEST_OF} {reuse_flag} --judge-model gemini-2.5-flash --sleep 0.2

assert os.path.exists(GEM), 'gemini responder did not write output -> read the traceback above'
preds = json.load(open(GEM))
assert len(preds) == 80, f'expected 80 Blind-A entries, got {len(preds)} -- DO NOT submit'
print('[blindA-207] entries:', len(preds), '| e5 w1.5 + reranker k=100 + flash-lite responder')
print('sample response:\n', preds[0]['predicted_response'][:400])
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-207-e5replace-k100-litegemini.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(GEM, arcname='prediction.json')   # 'prediction.json' at zip ROOT for CodaBench
print('[blindA-207] submission zip ready:', zip_path)
print('[blindA-207] upload to CodaBench; then log the score via scripts/blind_a_score_tracker.py')


In [ ]:
# 82-blindA-209) EXP-016 SHIP: config 209 = config 205 + use_doc_enriched (doc2query doc-enriched
# e5 dense channel). #8e-pg-confirm PASSED on dev: doc-enriched over (sasrec+pg) gave turn-1 nDCG@20
# +0.0359 with 12 net-new wall rescues -> ORTHOGONAL to pg -> ADD as the "ONE change", responder FROZEN.
# CAVEAT (RecSys review BLOCKER): #8e validated doc vs (sasrec+pg) ONLY; config 209 also runs ColBERT+CLAP,
# so doc x {colbert,clap} is UNCONFIRMED (its net-new wall rescues may overlap those channels; k=100
# crowding can demote good ids). Read the score against the +-0.05 Blind noise band. To confirm FIRST,
# run a full-stack (sasrec+pg+colbert+clap vs +doc) turn-1 check in nb74 before submitting.
# Responder = gemini-2.5-pro + top-n 1 + bo1 (EXACT match to the 0.50 file in REUSE_FROM, required for
# valid warm-start). --reuse-from regenerates ONLY rows whose top-1 track changed vs 0.50 -> the reused
# explanation is never stale (resolves the top_n=1 mismatch risk). Requires PLAID index + GEMINI_API_KEY.
import os, json, zipfile, datetime, subprocess
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY in Colab Secrets'
os.environ['GEMINI_RESPONDER_MODEL'] = 'gemini-2.5-pro'   # MATCH the 0.50 file's responder (valid reuse)
assert os.path.isdir(f'{CACHE_DIR}/retrieval_v2/colbert/plaid'), 'PLAID index missing -> run #82-build-index'

BEST_OF = 1

# FROZEN responder: reuse the known-good 0.50 responses; only rows whose top-1 track CHANGED hit Gemini.
# MUST be a file generated with the SAME responder config (pro / top-n 1 / bo1). '' -> regenerate all.
REUSE_FROM = '/content/drive/MyDrive/recsys2026_submissions/2026-06-13-blindA-205-flashrank-gemini-bo1.zip'
reuse_flag = f'--reuse-from {REUSE_FROM}' if REUSE_FROM else ''
MCRS = '/content/recsys2026/music-crs-baselines'
TID  = '209-union-sasrec-pg-colbert-claptext-docenriched-flashrank-llm-listwise-v5kto-blindA'
PRED = f'{MCRS}/exp/inference/blindset_A/{TID}.json'
GEM  = f'{MCRS}/exp/inference/blindset_A/{TID}_gemini.json'
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

# 0) ensure the doc2query doc-enriched e5 embeddings exist (config 209's doc-enriched channel, label
#    'doc-enriched-e5'). Built once by nb74 cell #8c; rebuilt here from the enriched parquet if missing.
PARQUET = '/content/drive/MyDrive/recsys2026_cache/enriched/catalog_enriched.parquet'
DOCENR = f'{CACHE_DIR}/dense_local/intfloat_multilingual-e5-large-instruct/doc-enriched-e5/track_embeddings.pkl'
if not os.path.exists(DOCENR):
    assert os.path.exists(PARQUET), f'enriched parquet missing ({PARQUET}) -> run nb74 #8b-enrich-catalog'
    print('[blindA-209] embedding doc-enriched catalog with e5 (one-time, fp16, ~10 min)...')
    subprocess.run(['python', 'scripts/embed_catalog.py',
                    '--model', 'intfloat/multilingual-e5-large-instruct', '--label', 'doc-enriched-e5',
                    '--doc-source', PARQUET, '--doc-col', 'doc_text',
                    '--batch-size', '256', '--dtype', 'auto', '--max-seq-len', '384',
                    '--cache-root', f'{CACHE_DIR}/dense_local'], cwd='/content/recsys2026', check=True)
assert os.path.exists(DOCENR), f'doc-enriched e5 embeddings not written to {DOCENR}'

# 1) full-stack retrieval: union (qwen-dense + doc-enriched-e5 + colbert + clap + pg + sasrec) -> flash
#    listwise ranker. --retrieval_only skips the qwen responder (discarded anyway by the step-2 swap).
if os.path.exists(PRED):
    print('[blindA-209] reusing existing track_ids:', PRED)
else:
    subprocess.run(['python', 'run_inference_blindset.py', '--tid', TID, '--batch_size', '8',
                    '--retrieval_only'], cwd=MCRS, check=False)
assert os.path.exists(PRED), 'inference did not write 209.json -> read the traceback above'

# 2) Gemini responder swap (pro; track_ids untouched; top-n 1 + bo1; reuse 0.50 for unchanged-top1 rows)
!cd /content/recsys2026 && python -u scripts/gemini_responder.py --cache-dir /content/drive/MyDrive/recsys2026_cache/responder --pred {PRED} --out {GEM} --dataset {BLIND_DATASET} --top-n 1 --best-of {BEST_OF} {reuse_flag} --judge-model gemini-2.5-flash --sleep 0.2

assert os.path.exists(GEM), 'gemini responder did not write output -> read the traceback above'
preds = json.load(open(GEM))
assert len(preds) == 80, f'expected 80 Blind-A entries, got {len(preds)} -- DO NOT submit'
print('[blindA-209] entries:', len(preds), '| config 205 + doc-enriched-e5 (w0.7) + pro responder reuse')
print('sample response:\n', preds[0]['predicted_response'][:400])
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-209-docenriched-gemini.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(GEM, arcname='prediction.json')   # 'prediction.json' at zip ROOT for CodaBench
print('[blindA-209] submission zip ready:', zip_path)
print('[blindA-209] upload to CodaBench; then log the score via scripts/blind_a_score_tracker.py')
